# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the [`mlcroissant`](https://mlcommons.github.io/croissant/) library to explore and process the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset, which provides mass spectrometry-based protein data for human extracellular vesicles (EVs).

### Dataset Source
The Croissant schema (RDF/JSON-LD) for the dataset is available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print high-level metadata/description
print(f"Dataset Title: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}")
print(f"Version: {getattr(metadata, 'version', '')}")
print(f"License: {getattr(metadata, 'license', '')}\n")
print("Authors:")
for author in getattr(metadata, 'author', []):
    print(f"  {author.get('@id', author)}")

## 2. Data Overview
Show available record sets, fields, and columns, referencing all by their `@id` for reproducibility.

In [ ]:
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No explicit record sets listed in top-level metadata. Attempting to extract from distribution or default data file.")
    # Try extracting from the main distribution
    dists = getattr(metadata, 'distribution', [])
    for dist in dists:
        print(f"Distribution @id: {dist.get('@id', dist)}")
    # Fallback: Use the main data file's record set (commonly @id = cr:RecordSet/0)
    print("A typical record set id for Croissant tabular data is: 'cr:RecordSet/0'")
    record_sets = ['cr:RecordSet/0']

print("\nRecord Set @ids Available:")
for idx, record_set in enumerate(record_sets):
    print(f"  {record_set}")

In [ ]:
record_set_id = record_sets[0]  # use the first available or fallback

# Show the fields (columns) of the selected record set
schema = dataset.schema

def list_fields_columns(rs_id):
    # Scan the schema for record set definitions
    rec_sets = schema.get('recordSet', [])
    if not rec_sets:
        return None
    for rs in rec_sets:
        if rs.get('@id') == rs_id:
            # Fields/columns can be under 'field', 'column', or their own @id list
            fields = rs.get('field', [])
            columns = rs.get('column', [])
            return fields, columns
    return None

fields, columns = list_fields_columns(record_set_id) or ([], [])
print(f"Fields in record set {record_set_id}:\n  {[f.get('@id', f) for f in fields]}")
print(f"Columns in record set {record_set_id}:\n  {[c.get('@id', c) for c in columns]}")

## 3. Data Extraction
Load data from the record set into a DataFrame for analysis, referencing by the `@id` as required.

In [ ]:
# Use only the available record sets
record_sets_to_read = record_sets
dataframes = {}

for rs in record_sets_to_read:
    print(f"Loading records from record set {rs}...")
    try:
        records_iter = dataset.records(record_set=rs)
        df = pd.DataFrame(records_iter)
        if not df.empty:
            dataframes[rs] = df
            print(f"Loaded dataframe for {rs} with columns: {df.columns.tolist()}")
        else:
            print(f"No data found for record set {rs}.")
    except Exception as e:
        print(f"Error loading record set {rs}: {e}")

# Select the first, as main
main_rs = record_sets_to_read[0]
if main_rs in dataframes:
    print("\nSample rows from main record set:")
    display(dataframes[main_rs].head())
else:
    print("No records loaded for primary record set.")

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group data using numeric and categorical fields, all referenced by their `@id`.

In [ ]:
# Choose numeric and group fields by examining columns (assuming typical proteomics table)
df = dataframes.get(main_rs, pd.DataFrame())
print("Columns available for EDA:", df.columns.tolist())

# --- Pick field names for numeric analysis (by @id if available) ---
# Example: Use the abundance or MW column
numeric_candidate_fields = [
    col for col in df.columns if ('abundance' in col.lower() or 'mw' in col.lower() or 'count' in col.lower() or 'intensity' in col.lower())
]
if numeric_candidate_fields:
    numeric_field = numeric_candidate_fields[0]
else:
    numeric_field = df.select_dtypes(include='number').columns[0] if not df.empty else None

# Example group by 'description' or 'sample' column if it exists
group_candidate_fields = [col for col in df.columns if ('sample' in col.lower() or 'description' in col.lower())]
group_field = group_candidate_fields[0] if group_candidate_fields else None

if numeric_field:
    # Clean up non-numeric values if necessary
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    print(f"\nFiltering records where {numeric_field} > 10 (if numeric)...")
    filtered_df = df[df[numeric_field] > 10].copy()

    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print("\nNormalized values:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize the normalized distributions and possible sample-wise grouping.

In [ ]:
# Visualize if columns and filtered data exist
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} (>10)")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to programmatically load, examine, and process a Croissant-annotated mass spectrometry protein dataset. By referencing all record sets and fields by their `@id`, this process ensures reproducibility and precise mapping to schema-defined data structures. You can further customize the steps to expand analyses or visualize relationships between additional fields as needed.